In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
df = pd.read_csv("house_data.csv")

df.head()

In [ ]:
#df['area_type'].unique()
df.info()

In [ ]:
df['area_type'].unique()


In [ ]:
df['availability'].unique()


In [ ]:
df['location'].unique()

In [ ]:
df['size'].unique()

In [ ]:
df['society'].unique()

In [ ]:
df = df.drop(['area_type','availability','balcony','society'],axis=1)
df.head()

In [ ]:
df.info()

In [ ]:

df.isnull().sum()

In [ ]:
df['location'].value_counts()

In [ ]:
df['location']= df['location'].fillna('Sarjapur  Road')
df.isnull().sum()

In [ ]:
df['size'].value_counts()

In [ ]:
df['size'].value_counts()
df['size'] = df['size'].fillna('2 BHK')


In [ ]:
med_bath=df['bath'].median()
med_bath
df['bath']=df['bath'].fillna(med_bath)

In [ ]:
df.drop_duplicates(inplace=True)

In [119]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6047 entries, 0 to 6046
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   location     6047 non-null   object 
 1   total_sqft   6047 non-null   float64
 2   bath         6047 non-null   float64
 3   price        6047 non-null   float64
 4   bhk          6047 non-null   int64  
 5   encoded_loc  6047 non-null   int64  
dtypes: float64(3), int64(2), object(1)
memory usage: 283.6+ KB


In [ ]:
df["location"].value_counts()
loc = df["location"].value_counts()
loc_less_than_10 = loc[loc<=10]

df["location"] = df["location"].apply(lambda x: "others" if x in loc_less_than_10 else x)
df["location"].value_counts()

In [ ]:
out = [int(i.split()[0]) for i in df["size"]]
df["bhk"] = out
df

In [ ]:
def clean_sqft(sqft):     
    l = sqft.split("-")
    if len(l)==2:
        return float(l[0])+float(l[1])/2
    try:
       return float(l[0]) 
    except:
        return None
df["total_sqft"] = df["total_sqft"].apply(clean_sqft)

df["total_sqft"] = df["total_sqft"].fillna(round(df["total_sqft"].mean()))

In [ ]:
df["price_per_sqft"] = df["price"]*100000/df["total_sqft"]
df

In [ ]:
df = df[df["total_sqft"]/df["bhk"]>=300]
df

In [ ]:
df = df[df['bhk']<=6]
df = df[df['bath']< df["bhk"]+2]

In [ ]:
q1 = df["price_per_sqft"].quantile(0.25)
q3 = df["price_per_sqft"].quantile(0.75)

IQR = q3-q1

lower=  q1-0.5*IQR
upper=  q1+0.5*IQR

df= df[(df["price_per_sqft"]>=lower) & (df["price_per_sqft"]<=upper)]


In [114]:
df.reset_index(inplace=True)
df = df.drop(['index','size','price_per_sqft'], axis= 1)

In [115]:
from sklearn.preprocessing import LabelEncoder, StandardScaler, Normalizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error

In [118]:
encoder = LabelEncoder()
df['encoded_loc'] = encoder.fit_transform(df["location"])
df.head()

,location,total_sqft,bath,price,bhk,encoded_loc
0,Electronic City Phase II,1056.0,2.0,39.07,2,66
1,Chikka Tirupathi,2600.0,5.0,120.00,4,51
2,Uttarahalli,1440.0,2.0,62.00,3,197
3,Kothanur,1200.0,2.0,51.00,2,133
4,Whitefield,1170.0,2.0,38.00,2,205


In [120]:
X = df.drop(["location","price"],axis= 1)
y = df.price

In [121]:
Xtrain,Xtest,ytrain,ytest = train_test_split(X,y,test_size=0.3,random_state=42)

In [123]:
model = RandomForestRegressor(random_state=42)
params = {
    "n_estimators":[100,150,200,250,300],
    "max_depth":[3,4,5,6,7]
}

grid = GridSearchCV(estimator=model,param_grid= params,cv= 5)
grid.fit(Xtrain,ytrain)

print("Best params",grid.best_params_)
print("best score",grid.best_score_)

Best params {'max_depth': 7, 'n_estimators': 150}
best score 0.8592595636406557


In [124]:
ypred = grid.predict(Xtest)
ypred

array([ 71.64251439, 120.61629384,  45.41678315, ...,  50.4735711 ,
        38.35864064,  46.82059326], shape=(1815,))

In [125]:
print("training eff:", grid.score(Xtrain,ytrain))
print("testingng eff:", grid.score(Xtest,ytest))

training eff: 0.9090396402727485
testingng eff: 0.8553288807880028


In [126]:
df.to_csv("cleaned_df.csv")

In [127]:
import joblib
with open("RF_model.joblib","wb") as file:
    joblib.dump(grid,file)